In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

1. Загрузите данные из файла data-logistic.csv.
Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [10]:
df = pd.read_csv("data-logistic.csv", header=None)
y = df[0]
X = df.loc[:, 1:]

2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска.
Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!

In [15]:

def calc_w1(X, y, w1, w2, k, C):
    x1 = X.iloc[:, 0]
    z = w1 * x1 + w2 * X.iloc[:, 1]
    s = (y * x1 * (1 - 1/(1 + np.exp(-y*z)))).mean()
    return w1 + k * s - k * C * w1

def calc_w2(X, y, w1, w2, k, C):
    x2 = X.iloc[:, 1]
    z = w1 * X.iloc[:, 0] + w2 * x2
    s = (y * x2 * (1 - 1/(1 + np.exp(-y * z)))).mean()
    return w2 + k * s - k * C * w2

3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии.
Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

In [19]:
def grad_desc(X, y, w1=0, w2=0, k=0.1, C=0, prec=1e-5, max_iter=10000):
    for _ in range(max_iter):
        w1_old, w2_old = w1, w2

        # Векторизованный расчет градиента
        z = w1 * X.iloc[:, 0] + w2 * X.iloc[:, 1]
        sig = 1 / (1 + np.exp(-y * z))

        w1 += k * (y * X.iloc[:, 0] * (1 - sig)).mean() - k * C * w1
        w2 += k * (y * X.iloc[:, 1] * (1 - sig)).mean() - k * C * w2

        if np.hypot(w1 - w1_old, w2 - w2_old) < prec:
            break
    return w1, w2

4. Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5).
Рекомендуется ограничить сверху число итераций десятью тысячами.

In [ ]:
w1, w2 = grad_desc(X, y)
w1_reg, w2_reg = grad_desc(X, y, C=10.0)

5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании?
Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом. Для этого воспользуйтесь сигмоидной функцией:
a(x) = 1 / (1 + exp(-w1 * x1 - w2 * x2)).

In [21]:
def a(X, w1, w2):
    return 1 / (1 + np.exp(-w1 * X[1] - w2 * X[2]))

y_proba = a(X, w1, w2)
y_proba_reg = a(X, w1_reg, w2_reg)

auc = roc_auc_score(y, y_proba)
auc_reg = roc_auc_score(y, y_proba_reg)

print(f"{auc:.3f} {auc_reg:.3f}")

0.927 0.936
